In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: LBCO, HRPT

This basic example is designed to show how Rietveld refinement can be
performed when both the crystal structure and experiment parameters
are defined using CIF files.

For this example, constant-wavelength neutron powder diffraction data
for La0.5Ba0.5CoO3 from HRPT at PSI is used.

The example is intended for users who are already familiar with the
EasyDiffraction library and want to quickly get started with a basic
refinement.

It is also useful for those who want to see how constraints can be
applied to highly correlated parameters. For a more detailed
explanation of the code, please refer to the other tutorials.

## Import Library

In [2]:
import easydiffraction as ed

## Step 1: Define Project

In [3]:
# Create minimal project without name and description
project = ed.Project()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 2: Define Crystal Structure

In [4]:
# Download CIF file from repository
structure_path = ed.download_data(id=1, destination='data')

Getting data...


Data #1: La0.5Ba0.5CoO3 (crystal structure)


✅ Data #1 downloaded to 'data/ed-1.cif'


In [5]:
# Add structure from downloaded CIF
project.structures.add_from_cif_path(structure_path)

## Step 3: Define Experiment

In [6]:
# Download CIF file from repository
expt_path = ed.download_data(id=2, destination='data')

Getting data...


Data #2: La0.5Ba0.5CoO3, HRPT (PSI), 300 K


✅ Data #2 downloaded to 'data/ed-2.cif'


In [7]:
# Add experiment from downloaded CIF
project.experiments.add_from_cif_path(expt_path)

## Step 4: Perform Analysis (no constraints)

In [8]:
# Start refinement. All parameters, which have standard uncertainties
# in the input CIF files, are refined by default.
project.analysis.fit()

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit (reduced χ²) change:


,iteration,χ²,improvement [%]
1,1,165.50,
2,18,132.82,19.7% ↓
3,24,62.80,52.7% ↓
4,40,48.63,22.6% ↓
5,41,18.58,61.8% ↓
6,57,15.15,18.5% ↓
7,58,7.00,53.8% ↓
8,76,4.17,40.4% ↓
9,93,1.39,66.7% ↓
10,111,1.29,7.0% ↓


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🏆 Best goodness-of-fit (reduced χ²) is 1.29 at iteration 138


✅ Fitting complete.


In [9]:
# Show fit results summary
project.analysis.display.fit_results()

Fit results


✅ Success: True


⏱️ Fitting time: 11.30 seconds


📏 Goodness-of-fit (reduced χ²): 1.29


📏 R-factor (Rf): 5.63%


...

📈 Fitted parameters:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,start,fitted,uncertainty,units,change
1,lbco,cell,,length_a,3.8800,3.8909,0.0000,Å,0.28 % ↑
2,lbco,atom_site,La,b_iso,0.5000,0.5079,46.3209,Å²,1.58 % ↑
3,lbco,atom_site,Ba,b_iso,0.5000,0.5005,75.0815,Å²,0.11 % ↑
4,lbco,atom_site,Co,b_iso,0.5000,0.2373,0.0612,Å²,52.54 % ↓
5,lbco,atom_site,O,b_iso,0.5000,1.3934,0.0167,Å²,178.68 % ↑
6,hrpt,linked_phases,lbco,scale,10.0000,9.1347,0.0641,,8.65 % ↓
7,hrpt,peak,,broad_gauss_u,0.1000,0.0817,0.0031,deg²,18.35 % ↓
8,hrpt,peak,,broad_gauss_v,-0.1000,-0.1160,0.0067,deg²,16.04 % ↑
9,hrpt,peak,,broad_gauss_w,0.1000,0.1205,0.0033,deg²,20.49 % ↑
10,hrpt,peak,,broad_lorentz_y,0.1000,0.0844,0.0021,deg,15.58 % ↓


red uncertainty — exceeds the fitted value (poorly constrained)


In [10]:
# Show parameter correlations
project.plotter.plot_param_correlations()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 5: Perform Analysis (with constraints)

In [11]:
# As can be seen from the parameter-correlation plot, the isotropic
# displacement parameters of La and Ba are highly correlated. Because
# La and Ba share the same mixed-occupancy site, their contributions to
# the neutron diffraction pattern are difficult to separate, especially
# since their coherent scattering lengths are not very different.
# Therefore, it is necessary to constrain them to be equal. First we
# define aliases and then use them to create a constraint.
project.analysis.aliases.create(
    label='biso_La',
    param=project.structures['lbco'].atom_sites['La'].b_iso,
)
project.analysis.aliases.create(
    label='biso_Ba',
    param=project.structures['lbco'].atom_sites['Ba'].b_iso,
)
project.analysis.constraints.create(expression='biso_Ba = biso_La')

In [12]:
# Start refinement. All parameters, which have standard uncertainties
# in the input CIF files, are refined by default.
project.analysis.fit()

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit (reduced χ²) change:


,iteration,χ²,improvement [%]
1,1,1.29,
2,68,1.29,


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🏆 Best goodness-of-fit (reduced χ²) is 1.29 at iteration 61


✅ Fitting complete.


In [13]:
# Show fit results summary
project.analysis.display.fit_results()

Fit results


✅ Success: True


⏱️ Fitting time: 3.51 seconds


📏 Goodness-of-fit (reduced χ²): 1.29


📏 R-factor (Rf): 5.63%


...

📏 Weighted R-factor (wR): 4.39%


📈 Fitted parameters:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,start,fitted,uncertainty,units,change
1,lbco,cell,,length_a,3.8909,3.8909,0.0000,Å,0.00 % ↑
2,lbco,atom_site,La,b_iso,0.5079,0.5051,0.0278,Å²,0.55 % ↓
3,lbco,atom_site,Co,b_iso,0.2373,0.2374,0.0565,Å²,0.03 % ↑
4,lbco,atom_site,O,b_iso,1.3934,1.3934,0.0160,Å²,0.00 % ↑
5,hrpt,linked_phases,lbco,scale,9.1347,9.1348,0.0538,,0.00 % ↑
6,hrpt,peak,,broad_gauss_u,0.0817,0.0817,0.0031,deg²,0.00 % ↓
7,hrpt,peak,,broad_gauss_v,-0.1160,-0.1160,0.0066,deg²,0.03 % ↓
8,hrpt,peak,,broad_gauss_w,0.1205,0.1205,0.0032,deg²,0.02 % ↓
9,hrpt,peak,,broad_lorentz_y,0.0844,0.0844,0.0021,deg,0.01 % ↑
10,hrpt,instrument,,twotheta_offset,0.6229,0.6229,0.0010,deg,0.00 % ↑


In [14]:
# Show parameter correlations
project.plotter.plot_param_correlations()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# Show defined experiment names
project.experiments.show_names()

Defined experiments 🔬


['hrpt']


In [16]:
# Plot measured vs. calculated diffraction patterns
project.plotter.plot_meas_vs_calc(expt_name='hrpt', show_residual=True)